# Dashboard completo de encuesta

Esta versión agrega **más visualizaciones** y una breve interpretación para cada una.
Objetivo: descubrir patrones, no solo contar respuestas.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.rcParams["figure.figsize"]=(12,6)

archivo="Encuesta_TerracotaH_limpia.xlsx"
df=pd.read_excel(archivo)

salida=Path("graficas_dashboard")
salida.mkdir(exist_ok=True)

print("Filas:",len(df))
print("Columnas:",len(df.columns))
display(df.head())


In [ ]:

def interpretar(vc):
    top=vc.idxmax()
    pct=100*vc.max()/vc.sum()

    print("\nInterpretación:")
    print(f"- Mayor concentración en: {top} ({pct:.1f}%)")

    if pct>50:
        print("- Hay una tendencia dominante.")
    else:
        print("- No existe una respuesta claramente dominante.")

def barras(col,titulo):
    vc=df[col].dropna().value_counts()

    plt.figure(figsize=(11,6))
    plt.barh(vc.index.astype(str),vc.values)

    for i,v in enumerate(vc.values):
        plt.text(v,i,str(v))

    plt.title(titulo)
    plt.xlabel("Frecuencia")

    plt.tight_layout()
    plt.savefig(salida/f"{titulo}.png")

    plt.show()

    interpretar(vc)

def pastel(col,titulo):
    vc=df[col].dropna().value_counts()

    plt.figure(figsize=(8,8))
    plt.pie(vc,labels=vc.index,autopct="%1.1f%%")

    plt.title(titulo)

    plt.savefig(salida/f"{titulo}.png")

    plt.show()

    interpretar(vc)

def heat(c1,c2,titulo):

    tabla=pd.crosstab(df[c1],df[c2])

    plt.figure(figsize=(12,7))
    plt.imshow(tabla,cmap="Blues")

    plt.xticks(
        range(len(tabla.columns)),
        tabla.columns,
        rotation=45,
        ha="right"
    )

    plt.yticks(
        range(len(tabla.index)),
        tabla.index
    )

    for i in range(tabla.shape[0]):
        for j in range(tabla.shape[1]):
            plt.text(j,i,tabla.iloc[i,j],ha="center")

    plt.title(titulo)

    plt.tight_layout()
    plt.savefig(salida/f"{titulo}.png")

    plt.show()

    print("\nInterpretación:")
    print("- Las zonas con valores altos muestran respuestas que aparecen juntas.")


## 1. Composición de participantes

In [ ]:
pastel(df.columns[1],"Perfil de participantes")

## 2. Distribución de estrés

In [ ]:
barras(df.columns[2],"Nivel de estres")

## 3. Ranking de causas

In [ ]:
barras(df.columns[3],"Ranking de causas del estres")

## 4. Top 5 respuestas (Pareto)

In [ ]:

vc=df[df.columns[3]].value_counts().head(5)

ac=vc.cumsum()/vc.sum()*100

fig,ax=plt.subplots()

ax.bar(vc.index,vc.values)

ax2=ax.twinx()
ax2.plot(range(len(vc)),ac.values)

plt.xticks(rotation=30)

plt.title("Pareto causas principales")

plt.show()

print("Interpretación:")
print("- Permite ver si pocas causas explican la mayoría de respuestas.")


## 5. Estrés vs descuido

In [ ]:
heat(df.columns[2],df.columns[4],"Relacion estres-salud")

## 6. Higiene oral

In [ ]:

barras(df.columns[6],"Frecuencia cepillado")
barras(df.columns[7],"Uso hilo dental")


## 7. Comparación porcentual

In [ ]:

for c in [df.columns[6],df.columns[7]]:
    p=df[c].value_counts(normalize=True)*100

    plt.figure(figsize=(10,5))

    plt.bar(
        p.index.astype(str),
        p.values
    )

    plt.ylabel("%")

    plt.xticks(rotation=20)

    plt.title("Porcentaje - "+c[:40])

    plt.show()

print("Interpretación:")
print("- Facilita comparar hábitos independientemente del número total.")


## 8. Síntomas reportados

In [ ]:

c=df[df.columns[10]].dropna()

e=(
c.astype(str)
.str.split(";")
.explode()
.str.strip()
)

vc=e.value_counts()

plt.figure(figsize=(12,6))

plt.barh(vc.index,vc.values)

plt.title("Frecuencia de signos")

plt.show()

print("Interpretación:")
print("- Los síntomas con más frecuencia pueden ser foco de intervención.")


## 9. Histograma de respuestas por participante

In [ ]:

conteo=df.notna().sum(axis=1)

plt.figure()

plt.hist(conteo)

plt.title("Completitud respuestas")

plt.xlabel("Preguntas respondidas")

plt.show()


## 10. Matriz de relaciones (todas las preguntas categóricas)

In [ ]:

cat=df.select_dtypes(include="object")

mat=np.zeros((len(cat.columns),len(cat.columns)))

for i in range(len(cat.columns)):
    for j in range(len(cat.columns)):
        mat[i,j]=(
            pd.crosstab(cat.iloc[:,i],cat.iloc[:,j]).values.sum()
        )

plt.figure(figsize=(12,10))

plt.imshow(mat)

plt.xticks(
range(len(cat.columns)),
[c[:15] for c in cat.columns],
rotation=90
)

plt.yticks(
range(len(cat.columns)),
[c[:15] for c in cat.columns]
)

plt.title("Mapa general de preguntas")

plt.show()

print("Interpretación:")
print("- Sirve para detectar preguntas con patrones similares.")
